In [11]:
%load_ext autoreload
%autoreload 2
import os

if os.getcwd().endswith("notebooks"):
    os.chdir("..")

print(os.getcwd()) # should end in /medjudge-audit

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/Users/berniceyan/medjudge-audit


Clinical deep-dive: reading the worst disagreements

Rank the judge-physician disagreements by how many judge configs diverged from the physician label, attach the real clinical content, and read them. Output: a CSV you annotate by hand.

**Methodology note:** we *annotate* disagreements against an existing multi-physician ground truth and characterize them — we are **not** a sole grader producing new ground truth. The physician labels stay the reference; we audit where and why judges diverge.


In [12]:
import json
import hashlib
import pandas as pd

a = pd.read_json("results/grades_track_a.jsonl", lines=True)
a = a[a.grade.notna()].copy()
a["disagree"] = (a.grade.astype(bool) != a.physician_label.astype(bool)).astype(int)

# rank items by how many judge configs disagreed with the physician
agg = (a.groupby("item_id")
         .agg(n_configs=("disagree", "size"),
              n_disagree=("disagree", "sum"),
              physician_label=("physician_label", "first"),
              theme=("theme", "first"),
              prompt_id=("prompt_id", "first"))
         .reset_index())
agg["frac_disagree"] = (agg.n_disagree / agg.n_configs).round(3)
worst = agg.sort_values(["n_disagree", "frac_disagree"], ascending=False).head(50).copy()
print("top disagreements:", worst.n_disagree.value_counts().to_dict())


top disagreements: {3: 50}


In [13]:
# attach clinical content via the sha1(rubric)[:8] join back to meta_eval
def conv_text(msgs):
    return "\n\n".join(f"{m['role']}: {m['content']}" for m in msgs)

lookup = {}
for line in open("data/meta_eval.jsonl"):
    r = json.loads(line)
    rub = r["rubric"] if isinstance(r["rubric"], str) else json.dumps(r["rubric"])
    key = f'{r["completion_id"]}|{hashlib.sha1(rub.encode()).hexdigest()[:8]}'
    lookup[key] = {"criterion": rub, "completion": r["completion"],
                   "conversation": conv_text(r["prompt"]),
                   "physician_votes": r["binary_labels"], "category": r.get("category", "")}

for f in ["criterion", "completion", "conversation", "category", "physician_votes"]:
    worst[f] = worst.item_id.map(lambda i: lookup.get(i, {}).get(f, ""))

# per-judge verdicts, so you can see who dissented
votes = (a.assign(v=a.grade.astype(bool))
           .pivot_table(index="item_id", columns="judge_model", values="v", aggfunc="first"))
worst = worst.merge(votes, left_on="item_id", right_index=True, how="left")
print("content attached for", (worst.criterion != "").sum(), "of", len(worst), "cases")


content attached for 50 of 50 cases


In [14]:
# blank annotation columns for you to fill in
worst["who_is_right"] = ""   # judge / physician / ambiguous
worst["error_type"]   = ""   # clinical-nuance / ambiguous-criterion / label-noise / over-literal
worst["note"]         = ""   # 1-2 sentences

judges = sorted(a.judge_model.unique())
cols = (["item_id", "prompt_id", "theme", "category", "n_disagree", "n_configs",
         "frac_disagree", "physician_label", "physician_votes"] + judges +
        ["who_is_right", "error_type", "note", "criterion", "conversation", "completion"])
worst = worst[[c for c in cols if c in worst.columns]]
worst.to_csv("results/deepdive_cases.csv", index=False)
print(f"wrote results/deepdive_cases.csv - {len(worst)} cases, {worst.shape[1]} columns")
worst[["theme", "n_disagree", "physician_label", "criterion"]].head(10)


wrote results/deepdive_cases.csv - 50 cases, 18 columns


,theme,n_disagree,physician_label,criterion
24,cluster:communication_health-professional_tail...,3,False,"A health professional (""User"") is conversing w..."
52,cluster:communication_not-health-professional_...,3,False,"A non-health professional (""User"") is conversi..."
78,cluster:context_seeking_enough-context_precise,3,True,"A non-health professional (""User"") is conversi..."
91,cluster:hedging_no-uncertainty_seeks_context,3,True,Step 1: Does the response seek more context fr...
98,cluster:communication_not-health-professional_...,3,False,Step 1: Does the response contain factually in...
189,cluster:complex_responses_detailed_accuracy_he...,3,True,The response must be entirely factually accura...
205,cluster:hedging_any-reducible-uncertainty_hedges,3,False,Step 1. Does the response directly answer the ...
207,cluster:emergency_referrals_emergent_emergency...,3,False,Judge whether the completion from the assistan...
224,cluster:complex_responses_simple_appropriate,3,True,Multiple physicians have reviewed this convers...
252,cluster:communication_not-health-professional_...,3,False,Step 1: Does the response contain factually in...


## Annotating misalignments between physician graders and LLM judges

In `results/deepdive_cases.csv`
- For each row, read the `conversation`, the `completion`, and the `criterion`, looked at `physician_label`/`physician_votes` vs each judge column, then filled out:

1. **who_is_right**: judge / physician / ambiguous
2. **error_type** — build a small taxonomy inductively, e.g.:
  - `clinical-nuance` — judge missed a medical subtlety the physician caught
  - `ambiguous-criterion` — the rubric is genuinely open to interpretation
  - `label-noise` — the physician label looks wrong
  - `over-literal` — judge fixated on exact wording
3. **note**: 1-2 sentences on who's right and why.

Then tallied the the taxonomy — in particular, **what fraction of apparent "judge errors" are actually
label noise?**


In [ ]:
import pandas as pd

done = pd.read_csv("results/deep-dive-graded.csv", skiprows=1)
done = done[done.error_type.notna() & (done.error_type.astype(str).str.strip() != "")]
if len(done):
    print(f"annotated {len(done)}/50 cases\n")
    print(done.error_type.value_counts(), "\n")
    label_noise = (done.error_type == "label-noise").mean()
    print(f"fraction of disagreements that are actually label noise: {label_noise:.0%}")
else:
    print("no annotations yet - fill in error_type in results/deepdive_cases.csv first and rename as results/deep-dive-graded.csv")


annotated 50/50 cases

error_type
label-noise           25
ambiguous-criteria     9
clinical-nuance        7
over-literal           3
premature-closure      2
over-refusal           2
non-commitment         1
misdiagnosis           1
Name: count, dtype: int64 

fraction of disagreements that are actually label noise: 50%
